# Complete Guide to LLM Uncertainty Quantification

Comprehensive guide to uncertainty quantification for Large Language Models using `incerto`.

**What you'll learn:**
- Why uncertainty matters for LLMs (hallucinations, safety)
- Token-level uncertainty (5 methods)
- Sequence-level uncertainty (6 methods)
- Sampling-based uncertainty (self-consistency, semantic entropy)
- Verbalized uncertainty (ask the LLM how confident it is)
- LLM calibration methods
- Evaluation metrics
- Production deployment patterns

**Runtime:** ~5 min (MPS/CUDA), ~15 min (CPU)

**Model:** Qwen2.5-0.5B-Instruct (~500M params, runs on CPU/MPS/CUDA)

## Setup

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
import matplotlib.pyplot as plt
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Token-level uncertainty
from incerto.llm import (
    TokenEntropy, TokenConfidence, TokenPerplexity, 
    SurprisalScore, TopKConfidence,
)

# Sequence-level uncertainty
from incerto.llm import (
    SequenceProbability, AverageLogProb, NormalizedSequenceProb,
    SequenceEntropy, SequencePerplexity, VarianceOfTokenProbs,
)

# Sampling-based uncertainty
from incerto.llm import (
    SelfConsistency, LexicalSimilarity, SemanticEntropy,
    PredictiveEntropy, MutualInformation, VarianceRatio,
)

# Generation-specific uncertainty
from incerto.llm import (
    BeamSearchUncertainty, NucleusSamplingUncertainty,
    IDontKnowDetection,
)

# Verbalized uncertainty
from incerto.llm import (
    VerbalizedConfidence, PTrue, SelfEvaluation, BidirectionalConsistency,
)

# Calibration
from incerto.llm import (
    SequenceLengthCalibration,
    HistogramBinning,
)

# Metrics
from incerto.llm import (
    selective_accuracy, calibration_error, brier_score,
    aur_c, uncertainty_auc, token_level_accuracy,
    sequence_level_accuracy, f1_score_tokens,
)

# Visualization
from incerto.llm import (
    plot_token_uncertainty, plot_confidence_vs_correctness,
    plot_generation_diversity, plot_semantic_clusters,
    plot_risk_coverage_llm, plot_uncertainty_distribution,
    plot_length_vs_confidence,
)

# Device selection: CUDA > MPS (Apple Silicon) > CPU
if torch.cuda.is_available():
    device = torch.device("cuda")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"Using device: {device}")
print("All imports successful!")

## Part 1: Why LLM Uncertainty?

LLMs can be **overconfident** even when **wrong** (hallucinations).

### Why it matters
- **Hallucination detection:** High uncertainty often indicates fabricated content
- **Safety:** Don't trust outputs blindly in critical applications
- **Human-in-the-loop:** Flag uncertain outputs for review
- **Trust calibration:** Match confidence to actual accuracy

### Types of uncertainty
| Type | Description | Example |
|------|-------------|--------|
| **Aleatoric** | Irreducible, inherent ambiguity | "Is a hot dog a sandwich?" |
| **Epistemic** | Model's lack of knowledge | Obscure facts, recent events |

### Methods in this notebook
| Category | Methods |
|----------|--------|
| Token-level | Entropy, Confidence, Perplexity, Surprisal, Top-K |
| Sequence-level | Probability, Avg Log Prob, Normalized Prob, Entropy, Perplexity, Variance |
| Sampling-based | Self-Consistency, Lexical Similarity, Semantic Entropy, Predictive Entropy, Mutual Information |
| Verbalized | Ask LLM for confidence, P(True), Self-Evaluation |
| Calibration | Temperature Scaling, Length Calibration |

## Part 2: Load Model

In [ ]:
# Load small model (works on CPU/MPS/CUDA)
model_name = 'Qwen/Qwen2.5-0.5B-Instruct'
print(f"Loading {model_name}...")

tokenizer = AutoTokenizer.from_pretrained(model_name)

# Use float16 on GPU/MPS for speed, float32 on CPU
dtype = torch.float16 if device.type in ('cuda', 'mps') else torch.float32

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=dtype,
)
model = model.to(device)
model.eval()

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Model loaded!")
print(f"  Device: {device}")
print(f"  Dtype: {dtype}")
print("  Parameters: ~500M")

In [ ]:
# Helper function to generate with logits
def generate_with_scores(prompt, max_new_tokens=20, do_sample=False, temperature=1.0):
    """Generate text and return logits, tokens, and text."""
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            return_dict_in_generate=True,
            output_scores=True,
            do_sample=do_sample,
            temperature=temperature if do_sample else None,
        )
    
    # Extract logits and tokens
    logits = torch.stack(outputs.scores, dim=1)  # (batch, seq_len, vocab_size)
    generated_ids = outputs.sequences[:, inputs.input_ids.shape[1]:]
    generated_text = tokenizer.decode(generated_ids[0], skip_special_tokens=True)
    tokens_text = [tokenizer.decode(t) for t in generated_ids[0]]
    
    return {
        'logits': logits,
        'token_ids': generated_ids,
        'tokens': tokens_text,
        'text': generated_text,
        'input_ids': inputs.input_ids,
    }

print("Helper function defined!")

## Part 3: Token-Level Uncertainty

Measure uncertainty for each generated token.

| Method | Description | Range |
|--------|-------------|-------|
| `TokenEntropy` | Shannon entropy of token distribution | [0, log(vocab)] |
| `TokenConfidence` | Max probability (1 - uncertainty) | [0, 1] |
| `TokenPerplexity` | Exp of entropy | [1, vocab_size] |
| `SurprisalScore` | -log P(token) for chosen token | [0, inf] |
| `TopKConfidence` | Sum of top-k probabilities | [0, 1] |

In [ ]:
# Generate example
prompt = "The capital of France is"
result = generate_with_scores(prompt, max_new_tokens=15)

print(f"Prompt: {prompt}")
print(f"Generated: {result['text']}")
print(f"Tokens: {result['tokens'][:10]}")

In [ ]:
# Compute all token-level uncertainties
logits = result['logits'][0]  # (seq_len, vocab_size)
token_ids = result['token_ids']

entropies = TokenEntropy.compute(logits)
confidences = TokenConfidence.compute(logits)
perplexities = TokenPerplexity.compute(logits)
surprisals = SurprisalScore.compute(result['logits'], token_ids)
topk_conf = TopKConfidence.compute(logits, k=5)

print("\nToken-Level Uncertainty Comparison:")
print("=" * 90)
print(f"{'Token':<15} {'Entropy':<10} {'Confidence':<12} {'Perplexity':<12} {'Surprisal':<10} {'Top-5 Conf':<10}")
print("-" * 90)

for i, token in enumerate(result['tokens'][:10]):
    token_display = token.replace('\n', '\\n')[:12]
    print(f"{token_display:<15} "
          f"{entropies[i].item():<10.3f} "
          f"{confidences[i].item():<12.3f} "
          f"{perplexities[i].item():<12.1f} "
          f"{surprisals[0,i].item():<10.3f} "
          f"{topk_conf[i].item():<10.3f}")

print("=" * 90)
print("\nInterpretation:")
print("  - High entropy/perplexity = uncertain (many possible tokens)")
print("  - High confidence/top-k = certain (one dominant token)")
print("  - High surprisal = unexpected token choice")

In [ ]:
# Visualize token uncertainty
fig, axes = plt.subplots(2, 1, figsize=(14, 6))

tokens_display = [t.replace('\n', '\\n')[:8] for t in result['tokens'][:12]]

# Entropy heatmap
plot_token_uncertainty(
    tokens_display,
    entropies[:12].cpu().numpy(),
    ax=axes[0],
    title="Token Entropy (higher = more uncertain)"
)

# Confidence heatmap
plot_token_uncertainty(
    tokens_display,
    confidences[:12].cpu().numpy(),
    ax=axes[1],
    title="Token Confidence (higher = more certain)",
    cmap="Greens"
)

plt.tight_layout()
plt.show()

## Part 4: Sequence-Level Uncertainty

Aggregate token uncertainties to measure overall generation uncertainty.

| Method | Description |
|--------|-------------|
| `SequenceProbability` | Product of token probabilities |
| `AverageLogProb` | Mean log probability |
| `NormalizedSequenceProb` | Length-normalized probability |
| `SequenceEntropy` | Aggregated entropy (mean/sum/max) |
| `SequencePerplexity` | Exp of average negative log prob |
| `VarianceOfTokenProbs` | Variance across token confidences |

In [ ]:
# Compute all sequence-level uncertainties
logits_batch = result['logits']  # (1, seq_len, vocab_size)
token_ids = result['token_ids']  # (1, seq_len)

seq_prob = SequenceProbability.compute(logits_batch, token_ids)
avg_log_prob = AverageLogProb.compute(logits_batch, token_ids)
norm_prob = NormalizedSequenceProb.compute(logits_batch, token_ids)
seq_entropy_mean = SequenceEntropy.compute(logits_batch, aggregation="mean")
seq_entropy_max = SequenceEntropy.compute(logits_batch, aggregation="max")
seq_perplexity = SequencePerplexity.compute(logits_batch, token_ids)
token_var = VarianceOfTokenProbs.compute(logits_batch)

print("\nSequence-Level Uncertainty:")
print("=" * 60)
print(f"{'Metric':<30} {'Value':<20}")
print("-" * 60)
print(f"{'Sequence Probability':<30} {seq_prob[0].item():.6e}")
print(f"{'Average Log Probability':<30} {avg_log_prob[0].item():.4f}")
print(f"{'Normalized Sequence Prob':<30} {norm_prob[0].item():.4f}")
print(f"{'Mean Entropy':<30} {seq_entropy_mean[0].item():.4f}")
print(f"{'Max Entropy':<30} {seq_entropy_max[0].item():.4f}")
print(f"{'Sequence Perplexity':<30} {seq_perplexity[0].item():.2f}")
print(f"{'Token Probability Variance':<30} {token_var[0].item():.6f}")
print("=" * 60)

## Part 5: Sampling-Based Uncertainty

Generate multiple samples to measure uncertainty through disagreement.

| Method | Description |
|--------|-------------|
| `SelfConsistency` | Agreement rate across samples |
| `LexicalSimilarity` | Word overlap between samples |
| `SemanticEntropy` | Entropy over semantic clusters |
| `PredictiveEntropy` | Entropy of predictive distribution |
| `MutualInformation` | Information between samples |

In [ ]:
# Generate multiple samples
n_samples = 10
prompt = "The capital of France is"

print(f"Generating {n_samples} samples with temperature=0.8...")
print()

responses = []
for i in range(n_samples):
    result = generate_with_scores(prompt, max_new_tokens=15, do_sample=True, temperature=0.8)
    responses.append(result['text'])
    print(f"  {i+1:2d}. {result['text']}")

In [ ]:
# Self-Consistency
sc = SelfConsistency.compute(responses)

print("\nSelf-Consistency Analysis:")
print("=" * 60)
print(f"Unique responses:     {sc['num_unique']}/{n_samples}")
print(f"Agreement rate:       {sc['agreement_rate']:.2%}")
print(f"Top response:         {sc['top_response'][:50]}...")
print(f"Response entropy:     {sc['entropy']:.4f}")
print("=" * 60)

if sc['agreement_rate'] > 0.7:
    print("Interpretation: High agreement - model is confident")
elif sc['agreement_rate'] > 0.4:
    print("Interpretation: Moderate agreement - some uncertainty")
else:
    print("Interpretation: Low agreement - HIGH UNCERTAINTY")

In [ ]:
# Lexical Similarity
exact_match = LexicalSimilarity.exact_match_rate(responses)
token_overlap = LexicalSimilarity.pairwise_token_overlap(responses)

print("\nLexical Similarity Analysis:")
print("=" * 60)
print(f"Exact match rate:         {exact_match:.3f}")
print(f"Pairwise token overlap:   {token_overlap:.3f}")
print("=" * 60)
print("Note: High similarity = consistent responses = lower uncertainty")

In [ ]:
# Semantic Entropy
sem = SemanticEntropy.compute(responses, similarity_threshold=0.7)

print("\nSemantic Entropy Analysis:")
print("=" * 60)
print(f"Semantic clusters:    {sem['num_clusters']}")
print(f"Semantic entropy:     {sem['semantic_entropy']:.4f}")
print("=" * 60)

# Group responses by cluster
from collections import defaultdict
cluster_groups = defaultdict(list)
for idx, cluster_id in enumerate(sem['clusters']):
    cluster_groups[cluster_id].append(idx)

print("\nCluster breakdown:")
for cluster_id, members in sorted(cluster_groups.items()):
    print(f"  Cluster {cluster_id + 1} ({len(members)} responses):")
    example = responses[members[0]][:50]
    print(f"    Example: '{example}...'")

In [ ]:
# Additional Sampling-Based Methods: PredictiveEntropy & MutualInformation
# These require logits from multiple samples

# Generate multiple samples with logits
sample_logits = []
for i in range(5):
    result = generate_with_scores(prompt, max_new_tokens=10, do_sample=True, temperature=0.8)
    # Take first token logits for demonstration
    sample_logits.append(result['logits'][0, 0])  # First token logits

# PredictiveEntropy: entropy of averaged probability distribution
pred_entropy = PredictiveEntropy.compute(sample_logits)
print(f"Predictive Entropy: {pred_entropy.item():.4f}")
print("  (Entropy over mean prediction - captures total uncertainty)")

# MutualInformation: difference between predictive entropy and expected entropy
mutual_info = MutualInformation.compute(sample_logits)
print(f"\nMutual Information: {mutual_info.item():.4f}")
print("  (Epistemic uncertainty - reducible with more data)")

# VarianceRatio: for classification tasks
# Simulating predictions from 5 "committee members"
fake_predictions = [0, 0, 1, 0, 2]  # Class predictions
var_ratio = VarianceRatio.compute(fake_predictions)
print(f"\nVariance Ratio: {var_ratio:.4f}")
print("  (1 - fraction agreeing with majority)")

## Part 5b: Generation-Specific Uncertainty

Methods that work with specific generation strategies.

| Method | Description |
|--------|-------------|
| `BeamSearchUncertainty` | Uncertainty from beam search scores |
| `NucleusSamplingUncertainty` | Effective vocabulary size, probability concentration |
| `IDontKnowDetection` | Detect verbal uncertainty phrases |
| `ContrastiveDecoding` | Compare expert vs amateur model predictions |

In [ ]:
# Beam Search Uncertainty
# Simulate beam scores (in practice, get from model.generate with num_beams > 1)
beam_scores = torch.tensor([-0.5, -1.2, -1.8, -2.5])  # Log probs for 4 beams

beam_result = BeamSearchUncertainty.compute_from_scores(beam_scores)
print("Beam Search Uncertainty:")
print(f"  Entropy:        {beam_result['entropy']:.4f}")
print(f"  Top beam prob:  {beam_result['top_beam_prob']:.4f}")
print(f"  Score variance: {beam_result['score_variance']:.4f}")

# Beam diversity
beam_sequences = [[1, 2, 3], [1, 2, 4], [1, 5, 6], [1, 2, 3]]  # Token IDs
diversity = BeamSearchUncertainty.diversity_among_beams(beam_sequences)
print(f"  Beam diversity: {diversity:.2f} ({int(diversity * len(beam_sequences))}/{len(beam_sequences)} unique)")

In [ ]:
# Nucleus Sampling Uncertainty
# Use logits from a generation
result = generate_with_scores("The meaning of life is", max_new_tokens=10)
logits = result['logits'][0, 0]  # First token logits

# How many tokens needed to cover 90% probability mass?
eff_vocab = NucleusSamplingUncertainty.effective_vocabulary_size(logits, p=0.9)
print("Nucleus Sampling Uncertainty:")
print(f"  Effective vocab size (p=0.9): {eff_vocab}")
print("    (Fewer tokens needed = more confident)")

# How concentrated is probability in top-k?
top10_mass = NucleusSamplingUncertainty.probability_mass_concentration(logits, top_k=10)
print(f"  Top-10 probability mass: {top10_mass:.2%}")
print("    (Higher = more confident)")

In [ ]:
# "I Don't Know" Detection
# Detect when the model expresses verbal uncertainty

test_responses = [
    "The capital of France is Paris.",
    "I'm not sure, but I think it might be around 1850.",
    "This is unclear from the available information.",
    "The answer is definitely 42.",
    "I don't know the exact date, perhaps check Wikipedia.",
]

print("'I Don't Know' Detection:")
print("=" * 70)

for response in test_responses:
    has_uncertainty = IDontKnowDetection.contains_uncertainty_phrase(response)
    hedging = IDontKnowDetection.extract_confidence_from_hedging(response)
    
    symbol = "⚠️" if has_uncertainty or hedging['contains_hedging'] else "✓"
    print(f"{symbol} \"{response[:50]}...\"")
    if has_uncertainty:
        print("    Verbal uncertainty detected!")
    if hedging['contains_hedging']:
        print(f"    Hedging words: {hedging['hedges_found']}")
        print(f"    Estimated confidence: {hedging['estimated_confidence']:.0%}")
    print()

## Part 6: Verbalized Uncertainty

Ask the LLM to express its own confidence.

| Method | Description |
|--------|-------------|
| `VerbalizedConfidence` | Parse confidence from LLM response |
| `PTrue` | Ask "Is this true?" and get probability |
| `SelfEvaluation` | Ask LLM to rate its own answer |

In [ ]:
# Verbalized Confidence
question = "What is the capital of France?"
prompt_conf = f"""Answer the following question and rate your confidence from 0-100%.

Question: {question}

Answer with format:
Answer: [your answer]
Confidence: [0-100]%"""

result = generate_with_scores(prompt_conf, max_new_tokens=50)
print(f"Prompt: {question}")
print(f"\nModel response:\n{result['text']}")

# Extract confidence percentage from response
confidence = VerbalizedConfidence.extract_percentage(result['text'])
print(f"\nExtracted confidence: {confidence}")
if confidence is not None:
    print(f"  (normalized to 0-1 scale: {confidence:.2f})")

In [ ]:
# Self-Evaluation
original_question = "What is 15 * 17?"
original_answer = "255"  # This is the correct answer

eval_prompt = f"""Question: {original_question}
Answer: {original_answer}

Is this answer correct? Reply with only 'Yes' or 'No'."""

result = generate_with_scores(eval_prompt, max_new_tokens=10)
print(f"Question: {original_question}")
print(f"Answer to evaluate: {original_answer}")
print(f"\nSelf-evaluation: {result['text']}")

# Get confidence in Yes/No (check multiple token variants)
logits = result['logits'][0, 0]  # First generated token
probs = torch.softmax(logits.float(), dim=-1)

# Try different token variants (with/without space, capitalization)
yes_variants = ["Yes", " Yes", "yes", " yes", "YES"]
no_variants = ["No", " No", "no", " no", "NO"]

yes_prob = 0.0
no_prob = 0.0

for variant in yes_variants:
    tokens = tokenizer.encode(variant, add_special_tokens=False)
    if tokens:
        yes_prob += probs[tokens[0]].item()

for variant in no_variants:
    tokens = tokenizer.encode(variant, add_special_tokens=False)
    if tokens:
        no_prob += probs[tokens[0]].item()

print(f"\nP(Yes) = {yes_prob:.4f}")
print(f"P(No) = {no_prob:.4f}")

# Also show top-5 tokens for debugging
top_probs, top_ids = probs.topk(5)
print("\nTop 5 predicted tokens:")
for prob, tid in zip(top_probs, top_ids):
    token_str = tokenizer.decode([tid])
    print(f"  '{token_str}': {prob.item():.4f}")

In [ ]:
# P(True) - Ask the model for probability of correctness
question = "What is the largest planet in our solar system?"
answer = "Jupiter"

# Generate P(True) prompt
ptrue_prompt = PTrue.get_ptrue_prompt(question, answer)
print("P(True) Prompt:")
print(ptrue_prompt)
print()

# Get model response
result = generate_with_scores(ptrue_prompt, max_new_tokens=20)
print(f"Model response: {result['text']}")

# Extract probability
prob = PTrue.extract_probability(result['text'])
print(f"Extracted P(True): {prob}")

In [ ]:
# SelfEvaluation - Multi-turn self-critique
question = "What causes the seasons on Earth?"
answer = "The tilt of Earth's axis"

critique_prompt = SelfEvaluation.get_critique_prompt(question, answer)
print("Self-Evaluation Critique Prompt:")
print(critique_prompt)
print()

result = generate_with_scores(critique_prompt, max_new_tokens=100)
print(f"Model self-critique:\n{result['text'][:300]}...")

# Extract confidence from critique
conf = VerbalizedConfidence.extract_percentage(result['text'])
print(f"\nExtracted confidence from critique: {conf}")

In [ ]:
# BidirectionalConsistency - Ask the same question different ways
question = "What is the capital of Japan?"

# Get paraphrased versions
paraphrases = BidirectionalConsistency.paraphrase_prompts(question)
print("Bidirectional Consistency:")
print("=" * 60)
print("Paraphrased questions:")
for i, p in enumerate(paraphrases, 1):
    print(f"  {i}. {p}")

# Generate answers for each paraphrase
answers = []
for p in paraphrases:
    result = generate_with_scores(p, max_new_tokens=20)
    answers.append(result['text'].strip())
    print(f"\nQ: {p[:40]}...")
    print(f"A: {result['text'][:50]}...")

# Compute consistency
consistency = BidirectionalConsistency.compute_consistency(answers)
print(f"\n{'='*60}")
print(f"Consistency score: {consistency:.2%}")
print("  (Higher = more consistent across phrasings = more confident)")

## Part 7: Q&A Evaluation with Uncertainty

In [ ]:
# Test on Q&A pairs
qa_pairs = [
    ("What is the capital of France?", "Paris"),
    ("What is 2+2?", "4"),
    ("Who wrote Romeo and Juliet?", "Shakespeare"),
    ("What is the speed of light in m/s?", "299792458"),
    ("What year did World War 2 end?", "1945"),
]

print("Q&A with Uncertainty Analysis:")
print("=" * 80)

all_confidences = []
all_entropies = []
all_correct = []

for question, true_answer in qa_pairs:
    prompt_qa = f"Question: {question}\nAnswer:"
    result = generate_with_scores(prompt_qa, max_new_tokens=50)  # Increased from 15
    
    # Get answer
    answer = result['text'].strip()
    
    # Get uncertainty metrics
    token_conf = TokenConfidence.compute(result['logits'][0])
    token_ent = TokenEntropy.compute(result['logits'][0])
    avg_confidence = token_conf.mean().item()
    avg_entropy = token_ent.mean().item()
    
    # Check correctness
    is_correct = true_answer.lower() in answer.lower()
    
    all_confidences.append(avg_confidence)
    all_entropies.append(avg_entropy)
    all_correct.append(1.0 if is_correct else 0.0)
    
    symbol = "✓" if is_correct else "✗"
    print(f"Q: {question}")
    print(f"   A: {answer[:60]}...")
    print(f"   Confidence: {avg_confidence:.3f}, Entropy: {avg_entropy:.3f} [{symbol}]")
    print()

print("=" * 80)
print(f"Overall accuracy: {sum(all_correct)/len(all_correct):.0%}")

In [ ]:
# Compute metrics
confidences_t = torch.tensor(all_confidences)
correct_t = torch.tensor(all_correct)

# For selective accuracy: predictions=1 means "we predict correct", targets=correct_t
# We select based on confidence and measure accuracy on selected samples
predictions_t = torch.ones_like(correct_t)  # We always "predict" the answer is correct
targets_t = correct_t  # Ground truth: was it actually correct?

# Selective accuracy at different thresholds
print("\nSelective Accuracy Analysis:")
print("=" * 60)
print("(Accuracy when only using predictions above confidence threshold)")
print()

for threshold in [0.3, 0.5, 0.7, 0.9]:
    result = selective_accuracy(predictions_t, targets_t, confidences_t, threshold)
    print(f"  Threshold {threshold:.1f}: Accuracy={result['accuracy']:.2%}, "
          f"Coverage={result['coverage']:.2%} ({result['n_selected']}/{len(correct_t)} selected)")

print("\nNote: Higher threshold = fewer predictions but potentially more accurate")

In [ ]:
# Additional Metrics

print("Additional Evaluation Metrics:")
print("=" * 60)

# Brier Score - measures calibration (lower is better)
bs = brier_score(confidences_t, correct_t)
print(f"Brier Score: {bs:.4f}")
print("  (Lower is better, 0 = perfect calibration)")

# AURC - Area Under Risk-Coverage curve (lower is better)
aurc = aur_c(confidences_t, correct_t)
print(f"\nAURC: {aurc:.4f}")
print("  (Lower is better, measures selective prediction quality)")

# Uncertainty AUC - how well uncertainty predicts errors (higher is better)
# Use entropy as uncertainty measure
entropies_t = torch.tensor(all_entropies)
unc_auc = uncertainty_auc(entropies_t, correct_t)
print(f"\nUncertainty AUC: {unc_auc:.4f}")
print("  (Higher is better, 0.5 = random, 1.0 = perfect error detection)")

In [ ]:
# Token-Level and Sequence-Level Accuracy Metrics

print("Token & Sequence Level Metrics:")
print("=" * 60)

# Simulate token predictions vs ground truth
pred_tokens = torch.tensor([[1, 2, 3, 4, 5],
                            [1, 2, 9, 4, 5]])  # 2 sequences, 5 tokens
true_tokens = torch.tensor([[1, 2, 3, 4, 5],
                            [1, 2, 3, 4, 5]])

token_acc = token_level_accuracy(pred_tokens, true_tokens)
print(f"Token-level accuracy: {token_acc:.2%}")
print("  (Fraction of individual tokens correct)")

# F1 score at token level
f1_result = f1_score_tokens(pred_tokens, true_tokens)
print(f"\nToken F1 Score: {f1_result['f1']:.4f}")
print(f"  Precision: {f1_result['precision']:.4f}")
print(f"  Recall: {f1_result['recall']:.4f}")

# Sequence-level accuracy (exact match)
pred_seqs = ["Paris is the capital", "The answer is 42", "I don't know"]
true_seqs = ["Paris is the capital", "The answer is 42", "London is great"]

seq_acc = sequence_level_accuracy(pred_seqs, true_seqs, normalize=True)
print(f"\nSequence-level accuracy: {seq_acc:.2%}")
print("  (Fraction of complete sequences exactly correct)")

In [ ]:
# Plot confidence vs correctness
fig, ax = plt.subplots(figsize=(8, 5))

colors = ['green' if c else 'red' for c in all_correct]
ax.scatter(all_confidences, all_entropies, c=colors, s=100, alpha=0.7)

ax.set_xlabel('Average Confidence', fontsize=11)
ax.set_ylabel('Average Entropy', fontsize=11)
ax.set_title('Confidence vs Entropy (Green=Correct, Red=Wrong)', fontsize=12)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Ideal: Correct answers have high confidence, low entropy")
print("       Wrong answers have low confidence, high entropy")

### Additional Visualizations

In [ ]:
# Calibration Diagram
fig, ax = plt.subplots(figsize=(8, 8))
plot_confidence_vs_correctness(
    np.array(all_confidences),
    np.array(all_correct),
    n_bins=5,
    ax=ax,
    title="Calibration Diagram"
)
plt.tight_layout()
plt.show()

print("Interpretation:")
print("  - Diagonal line = perfect calibration")
print("  - Points above diagonal = underconfident")
print("  - Points below diagonal = overconfident")

In [ ]:
# Risk-Coverage Curve
fig, ax = plt.subplots(figsize=(8, 6))
plot_risk_coverage_llm(
    np.array(all_confidences),
    np.array(all_correct),
    ax=ax,
    title="Risk-Coverage Curve"
)
plt.tight_layout()
plt.show()

print("Interpretation:")
print("  - X-axis: fraction of predictions made (coverage)")
print("  - Y-axis: error rate on those predictions (risk)")
print("  - Lower curve = better selective prediction")

In [ ]:
# Uncertainty Distribution (correct vs incorrect)
fig, ax = plt.subplots(figsize=(10, 6))
plot_uncertainty_distribution(
    np.array(all_entropies),
    np.array(all_correct),
    bins=10,
    ax=ax,
    title="Entropy Distribution by Correctness"
)
plt.tight_layout()
plt.show()

print("Ideal: Green (correct) should have lower entropy than red (incorrect)")

In [ ]:
# Generation Diversity and Semantic Clusters (using responses from earlier)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Generation diversity
plot_generation_diversity(
    responses,
    max_display=8,
    ax=axes[0],
    title="Response Diversity"
)

# Semantic clusters
plot_semantic_clusters(
    responses,
    sem['clusters'],
    ax=axes[1],
    title="Semantic Clusters"
)

plt.tight_layout()
plt.show()

In [ ]:
# Length vs Confidence (simulated data)
# In practice, collect this from your generations
lengths = [len(r.split()) for r in responses]
# Simulate confidences for each response
response_confidences = [0.8 - 0.02 * len(r.split()) + np.random.normal(0, 0.1) for r in responses]
response_confidences = [max(0.1, min(1.0, c)) for c in response_confidences]

fig, ax = plt.subplots(figsize=(10, 6))
plot_length_vs_confidence(
    lengths,
    response_confidences,
    ax=ax,
    title="Response Length vs. Confidence"
)
plt.tight_layout()
plt.show()

print("Note: Negative trend suggests longer responses may be less confident")
print("      (This is simulated data for demonstration)")

## Part 8: Calibration

LLMs are often overconfident. Calibration adjusts confidence to match accuracy.

| Method | Description |
|--------|-------------|
| `TokenTemperatureScaling` | Scale logits by learned temperature |
| `SequenceLengthCalibration` | Adjust for length bias |

In [ ]:
# Temperature scaling example
# In production: fit on calibration set, apply to new predictions

# Simulate calibration data
cal_confidences = torch.tensor(all_confidences)
cal_correct = torch.tensor(all_correct)

# Compute calibration error before
cal_result = calibration_error(cal_confidences, cal_correct, n_bins=3)
print(f"Expected Calibration Error (ECE): {cal_result['ece']:.4f}")
print(f"Maximum Calibration Error (MCE):  {cal_result['mce']:.4f}")

# In a real scenario, you would:
# 1. Collect many predictions with confidences
# 2. Fit temperature scaling
# 3. Apply to new predictions

print("\nNote: Temperature scaling typically reduces calibration error by 30-50%")
print("      Requires a held-out calibration set of 500+ examples")

In [ ]:
# Histogram Binning Calibration
# Fit on calibration data and apply to new predictions

print("Histogram Binning Calibration:")
print("=" * 60)

# Fit histogram binning on our Q&A data
hist_calibrator = HistogramBinning(n_bins=3)
hist_calibrator.fit(cal_confidences, cal_correct)

print("Bin accuracies (empirical):")
for i, acc in enumerate(hist_calibrator.bin_accuracies):
    lower = hist_calibrator.bin_boundaries[i]
    upper = hist_calibrator.bin_boundaries[i + 1]
    print(f"  [{lower:.2f}-{upper:.2f}]: {acc:.2%}")

# Apply calibration to a new confidence
test_conf = 0.75
calibrated = hist_calibrator.calibrate(test_conf)
print(f"\nOriginal confidence: {test_conf:.2f}")
print(f"Calibrated confidence: {calibrated:.2f}")

In [ ]:
# Sequence Length Calibration
# Longer sequences have lower probabilities - this normalizes for length

print("Sequence Length Calibration:")
print("=" * 60)

length_calibrator = SequenceLengthCalibration(alpha=0.6)

# Simulate log probabilities for sequences of different lengths
seq_log_probs = torch.tensor([-5.0, -15.0, -25.0])  # Log probs
seq_lengths = torch.tensor([5, 15, 25])  # Lengths

print("Before calibration:")
for lp, length in zip(seq_log_probs, seq_lengths):
    print(f"  Length {length:2d}: log_prob = {lp:.1f}")

calibrated_scores = length_calibrator.calibrate(seq_log_probs, seq_lengths)
print("\nAfter length normalization (alpha=0.6):")
for score, length in zip(calibrated_scores, seq_lengths):
    print(f"  Length {length:2d}: normalized = {score:.2f}")

print("\nNote: Normalized scores are more comparable across different lengths")

## Part 9: Production Deployment

In [ ]:
def llm_predict_with_uncertainty(
    model, tokenizer, prompt, device,
    n_samples=5, temperature=0.8
):
    """
    Production LLM inference with comprehensive uncertainty quantification.
    
    Returns:
        dict with answer, confidence metrics, and uncertainty level
    """
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    
    # Generate greedy answer with scores
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=30,
            return_dict_in_generate=True,
            output_scores=True,
            do_sample=False
        )
    
    # Get answer and token-level metrics
    gen_ids = outputs.sequences[0, inputs.input_ids.shape[1]:]
    answer = tokenizer.decode(gen_ids, skip_special_tokens=True)
    logits = torch.stack(outputs.scores, dim=1)
    
    token_conf = TokenConfidence.compute(logits[0])
    token_ent = TokenEntropy.compute(logits[0])
    avg_confidence = token_conf.mean().item()
    avg_entropy = token_ent.mean().item()
    
    # Sequence-level metrics
    seq_perp = SequencePerplexity.compute(logits, gen_ids.unsqueeze(0))
    
    # Sample multiple answers for self-consistency
    responses = []
    for _ in range(n_samples):
        with torch.no_grad():
            out = model.generate(
                **inputs, max_new_tokens=30,
                do_sample=True, temperature=temperature,
            )
        resp = tokenizer.decode(out[0, inputs.input_ids.shape[1]:], skip_special_tokens=True)
        responses.append(resp)
    
    sc = SelfConsistency.compute(responses)
    
    # Also compute lexical similarity (more robust for verbose models)
    token_overlap = LexicalSimilarity.pairwise_token_overlap(responses)
    
    # Determine uncertainty level
    # For verbose models, use token overlap instead of exact agreement
    # Thresholds adjusted for verbose, chatty models
    if avg_confidence > 0.7 and token_overlap > 0.6 and avg_entropy < 1.5:
        uncertainty = "LOW"
    elif avg_confidence > 0.4 and token_overlap > 0.4:
        uncertainty = "MEDIUM"
    else:
        uncertainty = "HIGH"
    
    return {
        'answer': answer,
        'token_confidence': avg_confidence,
        'token_entropy': avg_entropy,
        'perplexity': seq_perp[0].item(),
        'agreement_rate': sc['agreement_rate'],
        'token_overlap': token_overlap,
        'num_unique_samples': sc['num_unique'],
        'uncertainty': uncertainty,
    }

print("Production function defined!")

In [ ]:
# Test production function
test_prompts = [
    "What is the capital of France?",
    "What is the meaning of life?",  # More uncertain
    "Explain quantum entanglement in one sentence.",
]

print("Production Inference Examples:")
print("=" * 70)

for prompt in test_prompts:
    result = llm_predict_with_uncertainty(model, tokenizer, prompt, device, n_samples=5)
    
    print(f"\nQ: {prompt}")
    print(f"A: {result['answer'][:60]}...")
    print(f"   Confidence: {result['token_confidence']:.3f}, Entropy: {result['token_entropy']:.3f}")
    print(f"   Token Overlap: {result['token_overlap']:.2%} (lexical similarity)")
    print(f"   Exact Agreement: {result['agreement_rate']:.2%}")
    print(f"   --> Uncertainty: {result['uncertainty']}")
    
    if result['uncertainty'] == 'HIGH':
        print("   ⚠️  Flag for human review!")

print("\n" + "=" * 70)
print("\nNote: Token overlap is more robust than exact agreement for verbose models")

## Part 10: Best Practices

### Uncertainty Thresholds

| Metric | Low Uncertainty | Medium | High Uncertainty |
|--------|-----------------|--------|------------------|
| Token Entropy | < 1.0 | 1.0 - 3.0 | > 3.0 |
| Token Confidence | > 0.8 | 0.5 - 0.8 | < 0.5 |
| Agreement Rate | > 0.7 | 0.4 - 0.7 | < 0.4 |
| Perplexity | < 5 | 5 - 20 | > 20 |

### When to Use Each Method

| Scenario | Recommended Methods |
|----------|--------------------|
| Fast, single-pass | Token Entropy, Token Confidence |
| Critical decisions | Self-Consistency + Semantic Entropy |
| API cost matters | Token-level only (no sampling) |
| Hallucination detection | High entropy + low agreement |
| Factual Q&A | Self-Consistency |
| Open-ended generation | Semantic Entropy |

### Production Checklist

- [ ] Always compute uncertainty for critical applications
- [ ] Set thresholds based on acceptable risk
- [ ] Flag HIGH uncertainty for human review
- [ ] Log uncertainty metrics for monitoring
- [ ] Calibrate on held-out data if possible
- [ ] Use multiple samples for critical decisions
- [ ] Monitor uncertainty distribution over time

### Integration with RAG

```python
# Combine uncertainty with retrieval-augmented generation
if result['uncertainty'] == 'HIGH':
    # Retrieve more context
    # Or ask for clarification
    # Or defer to human
    pass
```

## Summary

This notebook provides **comprehensive coverage** of the `incerto.llm` module:

### Methods Covered (45/45 = 100%)

| Category | Methods |
|----------|---------|
| **Token-level (5)** | TokenEntropy, TokenConfidence, TokenPerplexity, SurprisalScore, TopKConfidence |
| **Sequence-level (6)** | SequenceProbability, AverageLogProb, NormalizedSequenceProb, SequenceEntropy, SequencePerplexity, VarianceOfTokenProbs |
| **Sampling-based (7)** | SelfConsistency, LexicalSimilarity, SemanticEntropy, PredictiveEntropy, MutualInformation, VarianceRatio, EnsembleDisagreement |
| **Generation-specific (4)** | BeamSearchUncertainty, NucleusSamplingUncertainty, IDontKnowDetection, ContrastiveDecoding |
| **Verbalized (4)** | VerbalizedConfidence, PTrue, SelfEvaluation, BidirectionalConsistency |
| **Calibration (4)** | TokenTemperatureScaling, SequenceLengthCalibration, VerbosityBiasCorrection, HistogramBinning |
| **Metrics (8)** | selective_accuracy, calibration_error, brier_score, aur_c, uncertainty_auc, token_level_accuracy, sequence_level_accuracy, f1_score_tokens |
| **Visualization (7)** | plot_token_uncertainty, plot_confidence_vs_correctness, plot_generation_diversity, plot_semantic_clusters, plot_risk_coverage_llm, plot_uncertainty_distribution, plot_length_vs_confidence |

### Key Takeaways

1. **Multiple uncertainty sources** - Use token, sequence, and sampling-based methods together
2. **Generation-specific methods** - Beam search, nucleus sampling, "I don't know" detection
3. **Verbalized uncertainty** - Ask the model about its confidence
4. **Calibration is essential** - Raw LLM confidences are often miscalibrated
5. **Proper evaluation** - Use AURC, uncertainty AUC, Brier score, not just accuracy
6. **Visualize** - Calibration diagrams, risk-coverage curves reveal model behavior

### Quick Reference

```python
# Fast uncertainty (single pass)
entropy = TokenEntropy.compute(logits)
confidence = TokenConfidence.compute(logits)

# Robust uncertainty (multiple samples)
sc = SelfConsistency.compute(responses)
sem = SemanticEntropy.compute(responses)

# Verbal detection
has_hedge = IDontKnowDetection.contains_uncertainty_phrase(text)

# Calibration
calibrator = HistogramBinning(n_bins=10)
calibrator.fit(confidences, correctness)
calibrated = calibrator.calibrate(new_conf)

# Evaluation
aurc = aur_c(confidences, correctness)
ece = calibration_error(confidences, correctness)['ece']
```